In [1]:
# ============================================================
# STAGE 9D — REAL PRODUCTION MODEL PREDICTION
# ============================================================

import json
import sys
from pathlib import Path

import joblib
import pandas as pd


# ============================================================
# 1. PROJECT ROOT
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

print("=" * 70)
print("STAGE 9D — REAL PRODUCTION MODEL PREDICTION")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)


# ============================================================
# 2. BACKEND
# ============================================================

BACKEND_DIR = PROJECT_ROOT / "backend"

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BACKEND_DIR)
    )


# ============================================================
# 3. IMPORT FEATURE MAPPER
# ============================================================

from feature_mapper import (
    build_56_feature_vector
)

print(
    "\n✓ Feature mapper loaded"
)


# ============================================================
# 4. MODEL FILES
# ============================================================

MODEL_FILE = (
    PROJECT_ROOT
    / "models"
    / "final_instagram_engagement_model.joblib"
)

FEATURE_FILE = (
    PROJECT_ROOT
    / "models"
    / "final_model_features.json"
)

METADATA_FILE = (
    PROJECT_ROOT
    / "models"
    / "final_model_metadata.json"
)


print("\n" + "=" * 70)
print("PRODUCTION FILE VALIDATION")
print("=" * 70)

print(
    "\nModel:",
    MODEL_FILE.exists()
)

print(
    "Feature schema:",
    FEATURE_FILE.exists()
)

print(
    "Metadata:",
    METADATA_FILE.exists()
)


if not MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Model not found:\n{MODEL_FILE}"
    )

if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Feature schema not found:\n{FEATURE_FILE}"
    )

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Metadata not found:\n{METADATA_FILE}"
    )


# ============================================================
# 5. LOAD MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING PRODUCTION MODEL")
print("=" * 70)

model = joblib.load(
    MODEL_FILE
)

print(
    "\n✓ Model loaded"
)

print(
    "Model type:",
    type(model).__name__
)


# ============================================================
# 6. LOAD SCHEMA
# ============================================================

with open(
    FEATURE_FILE,
    "r",
    encoding="utf-8"
) as f:

    schema = json.load(f)


production_features = schema.get(
    "all_features",
    []
)

if not production_features:

    production_features = (
        schema.get(
            "categorical_features",
            []
        )
        +
        schema.get(
            "numeric_features",
            []
        )
    )


print(
    "\nExpected production features:",
    len(production_features)
)


# ============================================================
# 7. TEST INPUT
# ============================================================

TEST_IMAGE = (
    PROJECT_ROOT
    / "datasets"
    / "raw"
    / "instagram_data"
    / "img"
    / "insta1.jpg"
)


caption = (
    "Amazing sunset in Sri Lanka! 🌅✨ "
    "Such a beautiful evening by the beach."
)

hashtags = (
    "#srilanka #travel #sunset "
    "#photography #beach"
)


print("\n" + "=" * 70)
print("REAL USER INPUT")
print("=" * 70)

print(
    "\nCaption:",
    caption
)

print(
    "\nHashtags:",
    hashtags
)

print(
    "\nCategory:",
    "Travel"
)

print(
    "Account type:",
    "Creator"
)

print(
    "Image:",
    TEST_IMAGE
)


# ============================================================
# 8. BUILD EXACT 56 FEATURES
# ============================================================

print("\n" + "=" * 70)
print("BUILDING 56-FEATURE VECTOR")
print("=" * 70)

features = build_56_feature_vector(

    caption=caption,

    hashtags=hashtags,

    category="Travel",

    account_type="Creator",

    follower_count=125000,

    following_count=850,

    account_age_days=1450,

    verified_status=0,

    posting_frequency=4.0,

    average_historical_engagement=0.052,

    audience_growth_rate=0.018,

    account_activity_level=0.75,

    content_consistency=0.70,

    posting_hour=18,

    day_of_week="Saturday",

    posting_time_period="Evening",

    media_type="Image",

    has_location=1,

    sponsored=0,

    content_originality=0.75,

    content_quality_score=0.80,

    creator_activity_score=0.72,

    image_path=str(TEST_IMAGE)
)


print(
    "\nGenerated shape:",
    features.shape
)


# ============================================================
# 9. EXACT FEATURE ORDER
# ============================================================

features = features[
    production_features
]


print(
    "Reordered to production schema"
)

print(
    "Final shape:",
    features.shape
)


# ============================================================
# 10. FINAL FEATURE VALIDATION
# ============================================================

if features.shape != (1, 56):

    raise ValueError(
        f"Invalid production shape: "
        f"{features.shape}"
    )


if list(features.columns) != production_features:

    raise ValueError(
        "Feature order does not match "
        "production schema."
    )


if features.isna().sum().sum() > 0:

    raise ValueError(
        "Feature vector contains missing values."
    )


print(
    "✓ Exact 56-feature vector validated"
)


# ============================================================
# 11. PREDICTION
# ============================================================

print("\n" + "=" * 70)
print("RUNNING PRODUCTION MODEL")
print("=" * 70)


prediction = model.predict(
    features
)


prediction_label = str(
    prediction[0]
)


print(
    "\nPredicted class:",
    prediction_label
)


# ============================================================
# 12. PROBABILITIES
# ============================================================

probabilities = {}

if hasattr(
    model,
    "predict_proba"
):

    probability_array = (
        model.predict_proba(
            features
        )[0]
    )

    classes = model.classes_

    for cls, probability in zip(
        classes,
        probability_array
    ):

        probabilities[
            str(cls)
        ] = float(
            probability
        )


    print(
        "\nClass probabilities:"
    )

    for cls, probability in (
        probabilities.items()
    ):

        print(
            f"{cls:<10}: "
            f"{probability:.6f} "
            f"({probability * 100:.2f}%)"
        )


# ============================================================
# 13. CONFIDENCE
# ============================================================

if probabilities:

    confidence = max(
        probabilities.values()
    )

else:

    confidence = None


print(
    "\nPrediction:",
    prediction_label
)

if confidence is not None:

    print(
        "Confidence:",
        f"{confidence:.6f}"
    )

    print(
        "Confidence percentage:",
        f"{confidence * 100:.2f}%"
    )


# ============================================================
# 14. INPUT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("INPUT SUMMARY")
print("=" * 70)

summary_features = [
    "category",
    "account_type",
    "follower_count",
    "following_count",
    "caption_length",
    "word_count",
    "hashtag_count",
    "caption_sentiment",
    "has_image",
    "image_width",
    "image_height",
    "brightness",
    "contrast",
    "sharpness",
    "estimated_image_quality"
]


for feature in summary_features:

    if feature in features.columns:

        print(
            f"{feature:<35}: "
            f"{features.iloc[0][feature]}"
        )


# ============================================================
# 15. SAVE PREDICTION
# ============================================================

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "production"
    / "stage9d"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


result = {
    "prediction": prediction_label,
    "confidence": confidence,
    "probabilities": probabilities,
    "feature_count": int(
        features.shape[1]
    ),
    "model": type(model).__name__,
    "image_used": True
}


json_output = (
    OUTPUT_DIR
    / "stage9d_prediction.json"
)


with open(
    json_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        indent=4
    )


csv_output = (
    OUTPUT_DIR
    / "stage9d_56_features.csv"
)


features.to_csv(
    csv_output,
    index=False
)


# ============================================================
# 16. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("STAGE 9D RESULT")
print("=" * 70)

print(
    "\n✓ Production model loaded"
)

print(
    "✓ Feature mapper loaded"
)

print(
    "✓ Exact 56 features generated"
)

print(
    "✓ Feature order validated"
)

print(
    "✓ Real image processed"
)

print(
    "✓ Prediction completed"
)

print(
    "\nPrediction:",
    prediction_label
)

if confidence is not None:

    print(
        "Confidence:",
        f"{confidence * 100:.2f}%"
    )


print(
    "\nPrediction JSON:"
)

print(
    json_output
)

print(
    "\nFeature CSV:"
)

print(
    csv_output
)

print("\n" + "=" * 70)
print("STAGE 9D COMPLETED")
print("=" * 70)

STAGE 9D — REAL PRODUCTION MODEL PREDICTION

Project root:
D:\newwwwwwww\AiBasedInstagramPrediction

✓ Feature mapper loaded

PRODUCTION FILE VALIDATION

Model: True
Feature schema: True
Metadata: True

LOADING PRODUCTION MODEL

✓ Model loaded
Model type: Pipeline

Expected production features: 56

REAL USER INPUT

Caption: Amazing sunset in Sri Lanka! 🌅✨ Such a beautiful evening by the beach.

Hashtags: #srilanka #travel #sunset #photography #beach

Category: Travel
Account type: Creator
Image: D:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data\img\insta1.jpg

BUILDING 56-FEATURE VECTOR

Generated shape: (1, 56)
Reordered to production schema
Final shape: (1, 56)
✓ Exact 56-feature vector validated

RUNNING PRODUCTION MODEL

Predicted class: Medium

Class probabilities:
High      : 0.148180 (14.82%)
Low       : 0.186054 (18.61%)
Medium    : 0.665766 (66.58%)

Prediction: Medium
Confidence: 0.665766
Confidence percentage: 66.58%

INPUT SUMMARY
category                